# 3.1.3 — Tilde tarafının altı Cartan ilişkisinin V üzerinde ispatı

**Hedef.** Standart taraftaki yedi Cartan ilişkisinin (3.1.2) Koszul/tilde
karşılığı olan altı operatör ilişkisini, jenerik bir multivector $V$
üzerinde, sırayla, sistemin içinde adım-adım kapatmak.

Tilde operatörleri Poisson manifold $(M,\pi)$ üzerinde:
$\tilde{\iota}_\omega$ multivectoru $\omega$ ile **bir slot**
sıkıştırır; $\tilde{d}_\pi$ Schouten–Nijenhuis $\tilde{d}_\pi V :=
[\pi, V]_{\mathrm{SN}}$ ile bir derece **yükseltir**;
$\tilde{\mathcal{L}}_\omega := \tilde{d}\,\tilde{\iota}_\omega +
\tilde{\iota}_\omega\,\tilde{d}$ Lie türevidir.

| # | İlişki | Anlamı | Poisson? |
|---|---|---|---|
| 1 | $\tilde{\iota}_\omega\,\tilde{\iota}_\eta + \tilde{\iota}_\eta\,\tilde{\iota}_\omega = 0$ | iç çarpımlar anti-commute | hayır |
| 2 | $\tilde{d}^2 V = 0$ | dış-türevin nilpotensi | **evet** |
| 3 | $\tilde{\mathcal{L}}_\omega V = \tilde{d}\,\tilde{\iota}_\omega V + \tilde{\iota}_\omega\,\tilde{d} V$ | Cartan sihirli formülü | hayır |
| 4 | $[\tilde{\mathcal{L}}_\alpha, \tilde{\iota}_\beta] = \tilde{\iota}_{[\alpha,\beta]_K}$ | Lie/iç çarpım komütatörü | hayır |
| 5 | $[\tilde{\mathcal{L}}_\alpha, \tilde{d}] V = 0$ | Lie ile $\tilde{d}$ commute | **evet** |
| 6 | $[\tilde{\mathcal{L}}_\alpha, \tilde{\mathcal{L}}_\beta] V = \tilde{\mathcal{L}}_{[\alpha,\beta]_K} V$ | Lie türevlerinin komütatörü | **evet** |

İspat zincirleri `display_chain` ile LaTeX olarak basılır.


## Strateji

Standart taraftan farklı olarak burada **tek engine** kullanıyoruz:
`KoszulProblem.tilde_intrinsic_engine()` 28 kuralı ile birlikte
döner. Bu engine içinde Faz 14 boyunca eklenen kurallar var:

- **14.A–E** — iç çarpım, $\tilde{d}$, $\tilde{\mathcal{L}}$ için
  intrinsik açılım (slot Leibniz, alternating, anchor-anti-symmetry,
  vb.) + 6 closure aksiyomu.
- **14.F** — Cartan sihirli formülünün defining sürümü
  (`TildeCartanMagicDefinition`).
- **14.G** — `WrappedPairingAnchorAntisymmetryDefinition` ile
  `TildeSnJacobiResidueDefinition`: ilk kural bracket halkasında
  $\langle \pi^\sharp a, b \rangle + \langle \pi^\sharp b, a \rangle = 0$
  ile beraber gelen ikili iptalleri sıfırlar; ikincisi
  $[\pi,\pi]_{\mathrm{SN}}(\alpha,\beta,\gamma) = 0$ Schouten–Nijenhuis
  Jacobi engelinin 5-terimli kanonik formunu yakalar (Poisson-gated).

İspat tek çağrıdan geçiyor: `prob.prove_tilde_cartan(LHS, RHS,
etas=(η_1, …, η_q))`. Sistem önce bilgili bir sırada multi-evaluation
açar, sonra engine'i fix-point'e kadar koşturur. Sıfıra inerse
`ProofChain`, kalırsa `ProofFailure`.

İlişki **2/5/6** Poisson varsayımına ihtiyaç duyar: bu durumda
ilgili hücrelerden önce `prob.assume_poisson()` çağrılır ve engine
Poisson-gated kuralları aktive eder.


In [1]:
# Notebook doğrudan açıldığında jacopy'ı import edilebilir hâle getirir.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401


## 1. Kurulum — semboller, problem, yardımcı fonksiyon

Aşağıda kullanılan semboller:

- $\pi$: Poisson bivektörü (jenerik); ilişki 2/5/6'dan önce
  `prob.assume_poisson()` ile $[\pi,\pi]_{\mathrm{SN}} = 0$ açılır.
- $V$: jenerik multivector; her ilişki kendi doğal SN-derecesinde alır
  (1-VF veya 2-VF).
- $\omega, \eta, \alpha, \beta, \xi$: jenerik 1-form'lar
  (`Graded(degree=1)`).

Her ilişki için ayrı bir `KoszulProblem` örneği kuruyoruz; `_build`
yardımcısı (testlerdeki `_build_problem`'in birebir karşılığı)
problemi, registry'yi, $\pi$'yi, $V$'yi ve formları döndürür.


In [2]:
from jacopy.algebra.derivation import Act
from jacopy.calculus.tilde import (
    TildeExteriorDerivative,
    TildeInteriorProduct,
    TildeLieDerivative,
)
from jacopy.core.expr import Integer, Neg, Sum, Symbol
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.display.jupyter import display_chain
from jacopy.library.koszul_problem import KoszulProblem


def _build(form_names=("ω", "η"), V_degree=1):
    """Test'lerdeki ``_build_problem`` ile birebir aynı kurulum."""
    reg = PropertyRegistry()
    pi = Symbol("π")
    forms = tuple(Symbol(n) for n in form_names)
    for f in forms:
        reg.declare(f, Graded(degree=1))
    V = Symbol("V")
    reg.declare(V, Graded(degree=V_degree))
    prob = KoszulProblem(
        pi,
        forms,
        registry=reg,
        multivectors=((V, V_degree),),
    )
    return (prob, reg, pi, V) + forms


def prove(label, prob, lhs, rhs, etas):
    """Tek çağrı arayüzü; adım sayısını basıp ProofChain döndürür."""
    chain = prob.prove_tilde_cartan(lhs, rhs, etas=etas)
    print(f"{label} → {len(chain)} adımda kapandı")
    return chain


_probe, *_ = _build()
print("Engine kural sayısı:", len(_probe.tilde_intrinsic_engine().definitions))


Engine kural sayısı: 28


## 2. İlişki 1: $\tilde{\iota}$ anti-commute

$$
\tilde{\iota}_\omega\,\tilde{\iota}_\eta\,V \;+\;
\tilde{\iota}_\eta\,\tilde{\iota}_\omega\,V \;=\; 0.
$$

**Yapı.** Her $\tilde{\iota}$ bir form-slot tüketir; $V$'yi 2-vector
seçiyoruz ki sonuç hâlâ değerlendirilebilir bir slot bıraksın. Tek
1-form $\xi$ ile değerlendirip MultiEval'in alternating-kanonikalleşmesi
üzerinden iki terim toplamı sıfır olur:

$$
V(\omega, \eta, \xi) \;+\; V(\eta, \omega, \xi) \;=\; 0.
$$


In [3]:
prob, _, _, V, omega, eta = _build(V_degree=2)
xi = Symbol("ξ")
prob.registry.declare(xi, Graded(degree=1))

lhs = Sum.make(
    Act(TildeInteriorProduct(omega), Act(TildeInteriorProduct(eta), V)),
    Act(TildeInteriorProduct(eta), Act(TildeInteriorProduct(omega), V)),
)
chain = prove("(2-VF, eval ξ) ι̃_ω ι̃_η V + ι̃_η ι̃_ω V = 0",
              prob, lhs, Integer(0), etas=(xi,))
display_chain(chain)


(2-VF, eval ξ) ι̃_ω ι̃_η V + ι̃_η ι̃_ω V = 0 → 10 adımda kapandı


\begin{align*}
\left(\tilde{\iota}_{\omega}\!\left(\tilde{\iota}_{\eta}\!\left(V\right)\right) + \tilde{\iota}_{\eta}\!\left(\tilde{\iota}_{\omega}\!\left(V\right)\right)\right)\!\left(\xi\right) &\to \left(\tilde{\iota}_{\omega}\!\left(\tilde{\iota}_{\eta}\!\left(V\right)\right)\right)\!\left(\xi\right) + \left(\tilde{\iota}_{\eta}\!\left(\tilde{\iota}_{\omega}\!\left(V\right)\right)\right)\!\left(\xi\right) && \text{[MultiEval head linearity]\,(axiom)}\;\text{--- apply axiom: MultiEval head linearity} \\
\left(\tilde{\iota}_{\omega}\!\left(\tilde{\iota}_{\eta}\!\left(V\right)\right)\right)\!\left(\xi\right) &\to \left(\tilde{\iota}_{\eta}\!\left(V\right)\right)\!\left(\omega,\, \xi\right) && \text{[\ensuremath{\iota}̃\_\ensuremath{\omega} intrinsic: (\ensuremath{\iota}̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, …)]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}̃\_\ensuremath{\omega} intrinsic: (\ensuremath{\iota}̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, …)} \\
\left(\tilde{\iota}_{\eta}\!\left(V\right)\right)\!\left(\omega,\, \xi\right) &\to V\!\left(\eta,\, \omega,\, \xi\right) && \text{[\ensuremath{\iota}̃\_\ensuremath{\omega} intrinsic: (\ensuremath{\iota}̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, …)]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}̃\_\ensuremath{\omega} intrinsic: (\ensuremath{\iota}̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, …)} \\
V\!\left(\eta,\, \omega,\, \xi\right) &\to -V\!\left(\eta,\, \xi,\, \omega\right) && \text{[alternating canonicalize: bubble swap toward repr-sorted args]\,(axiom)}\;\text{--- apply axiom: alternating canonicalize: bubble swap toward repr-sorted args} \\
\left(\tilde{\iota}_{\eta}\!\left(\tilde{\iota}_{\omega}\!\left(V\right)\right)\right)\!\left(\xi\right) &\to \left(\tilde{\iota}_{\omega}\!\left(V\right)\right)\!\left(\eta,\, \xi\right) && \text{[\ensuremath{\iota}̃\_\ensuremath{\omega} intrinsic: (\ensuremath{\iota}̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, …)]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}̃\_\ensuremath{\omega} intrinsic: (\ensuremath{\iota}̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, …)} \\
\left(\tilde{\iota}_{\omega}\!\left(V\right)\right)\!\left(\eta,\, \xi\right) &\to V\!\left(\omega,\, \eta,\, \xi\right) && \text{[\ensuremath{\iota}̃\_\ensuremath{\omega} intrinsic: (\ensuremath{\iota}̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, …)]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}̃\_\ensuremath{\omega} intrinsic: (\ensuremath{\iota}̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, …)} \\
V\!\left(\omega,\, \eta,\, \xi\right) &\to -V\!\left(\eta,\, \omega,\, \xi\right) && \text{[alternating canonicalize: bubble swap toward repr-sorted args]\,(axiom)}\;\text{--- apply axiom: alternating canonicalize: bubble swap toward repr-sorted args} \\
V\!\left(\eta,\, \omega,\, \xi\right) &\to -V\!\left(\eta,\, \xi,\, \omega\right) && \text{[alternating canonicalize: bubble swap toward repr-sorted args]\,(axiom)}\;\text{--- apply axiom: alternating canonicalize: bubble swap toward repr-sorted args} \\
0\!\left(\xi\right) &\to 0 && \text{[MultiEval zero-head → 0]\,(axiom)}\;\text{--- apply axiom: MultiEval zero-head → 0} \\
\left(-V\!\left(\eta,\, \xi,\, \omega\right) - \left(-V\!\left(\eta,\, \xi,\, \omega\right)\right)\right) - 0 &\to 0 && \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)}
\end{align*}

## 3. İlişki 2: $\tilde{d}^2 = 0$ (Poisson)

$$
\tilde{d}\bigl(\tilde{d} V\bigr) \;=\; 0
\qquad \bigl(\pi \text{ Poisson, yani } [\pi,\pi]_{\mathrm{SN}} = 0\bigr).
$$

**Yapı.** Bu ilişki *operatör* seviyesinde
`TildeDSquaredPoissonDefinition` (Aux-5) closure aksiyomu ile kapanır.
Aksiyom Poisson varsayımına bağlıdır: `prob.assume_poisson()`
çağrılmazsa engine'da bir no-op kalır ve ispat başarısız olur — yani
sistem aslında "$\pi$ Poisson mı?" sorusunu da çözebilir. Aksiyom
multi-eval açılımından **önce** match olduğu için herhangi bir
non-empty `etas` tuple'ı çalışır; biz $(\eta, \xi)$ veriyoruz.


In [4]:
prob, _, _, V, eta, xi = _build(form_names=("η", "ξ"), V_degree=1)
prob.assume_poisson()  # [π,π]_SN = 0 açılır → Aux-5 aktif

lhs = Act(
    TildeExteriorDerivative(prob.pi),
    Act(TildeExteriorDerivative(prob.pi), V),
)
chain = prove("(1-VF, eval η,ξ) d̃(d̃ V) = 0  [π Poisson]",
              prob, lhs, Integer(0), etas=(eta, xi))
display_chain(chain)


(1-VF, eval η,ξ) d̃(d̃ V) = 0  [π Poisson] → 4 adımda kapandı


\begin{align*}
\tilde{d}\!\left(\tilde{d}\!\left(V\right)\right) &\to 0 && \text{[d̃² V = 0  (\ensuremath{\pi} Poisson) [\ensuremath{\pi}]]\,(axiom)}\;\text{--- apply axiom: d̃² V = 0  (\ensuremath{\pi} Poisson) [\ensuremath{\pi}]} \\
0\!\left(\eta,\, \xi\right) &\to 0 && \text{[MultiEval zero-head → 0]\,(axiom)}\;\text{--- apply axiom: MultiEval zero-head → 0} \\
0\!\left(\eta,\, \xi\right) &\to 0 && \text{[MultiEval zero-head → 0]\,(axiom)}\;\text{--- apply axiom: MultiEval zero-head → 0} \\
0 - 0 &\to 0 && \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)}
\end{align*}

## 4. İlişki 3: Cartan'ın sihirli formülü

$$
\tilde{\mathcal{L}}_\omega V \;=\; \tilde{d}\bigl(\tilde{\iota}_\omega V\bigr)
   \;+\; \tilde{\iota}_\omega\bigl(\tilde{d} V\bigr).
$$

**Yapı.** Tilde tarafında bu ilişki $\tilde{\mathcal{L}}$'nin
*tanımlayıcı* eşitliğidir (Faz 14.F'de
`TildeCartanMagicDefinition` olarak engine'a eklendi). 1-vector $V$ ve
tek 1-form $\eta$ ile değerlendirildiğinde her iki taraf bir fonksiyona
düşer; intrinsik kurallar + Aux-6 köprüsü çıplak
$\tilde{\iota}_\omega V$'yi `MultiEval(V, ω)` skalere çevirir.


In [5]:
prob, _, _, V, omega, eta = _build(V_degree=1)

lhs = Act(TildeLieDerivative(omega, prob.pi), V)
rhs = Sum.make(
    Act(TildeExteriorDerivative(prob.pi),
        Act(TildeInteriorProduct(omega), V)),
    Act(TildeInteriorProduct(omega),
        Act(TildeExteriorDerivative(prob.pi), V)),
)
chain = prove("(1-VF, eval η) L̃_ω V = d̃(ι̃_ω V) + ι̃_ω(d̃ V)",
              prob, lhs, rhs, etas=(eta,))
display_chain(chain)


(1-VF, eval η) L̃_ω V = d̃(ι̃_ω V) + ι̃_ω(d̃ V) → 11 adımda kapandı


\begin{align*}
\left(\tilde{\mathcal{L}}_{\omega}\!\left(V\right)\right)\!\left(\eta\right) &\to \left(\pi\sharp\!\left(\omega\right)\right)\!\left(V\!\left(\eta\right)\right) - V\!\left(\left[\omega,\, \eta\right]_{[\cdot,\cdot]_K[\pi]}\right) && \text{[L̃\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (L̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, …) − \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, …, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, …)]\,(axiom)}\;\text{--- apply axiom: L̃\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (L̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, …) − \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, …, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, …)} \\
\left[\omega,\, \eta\right]_{[\cdot,\cdot]_K[\pi]} &\to L_\pi\sharp(\omega)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\omega\right) - d\!\left(\langle \pi\sharp\!\left(\omega\right),\, \eta \rangle\right) && \text{[[\ensuremath{\alpha}, \ensuremath{\beta}]\_K = L\_\ensuremath{\rho}\ensuremath{\alpha}(\ensuremath{\beta}) − L\_\ensuremath{\rho}\ensuremath{\beta}(\ensuremath{\alpha}) − d\ensuremath{\langle}\ensuremath{\rho}\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} [[\ensuremath{\cdot},\ensuremath{\cdot}]\_K[\ensuremath{\pi}]]]\,(axiom)}\;\text{--- apply axiom: [\ensuremath{\alpha}, \ensuremath{\beta}]\_K = L\_\ensuremath{\rho}\ensuremath{\alpha}(\ensuremath{\beta}) − L\_\ensuremath{\rho}\ensuremath{\beta}(\ensuremath{\alpha}) − d\ensuremath{\langle}\ensuremath{\rho}\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} [[\ensuremath{\cdot},\ensuremath{\cdot}]\_K[\ensuremath{\pi}]]} \\
V\!\left(L_\pi\sharp(\omega)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\omega\right) - d\!\left(\langle \pi\sharp\!\left(\omega\right),\, \eta \rangle\right)\right) &\to V\!\left(L_\pi\sharp(\omega)\!\left(\eta\right)\right) - V\!\left(L_\pi\sharp(\eta)\!\left(\omega\right)\right) - V\!\left(d\!\left(\langle \pi\sharp\!\left(\omega\right),\, \eta \rangle\right)\right) && \text{[MultiEval arg slot linearity]\,(axiom)}\;\text{--- apply axiom: MultiEval arg slot linearity} \\
\tilde{d}\!\left(\tilde{\iota}_{\omega}\!\left(V\right)\right) &\to \tilde{d}\!\left(V\!\left(\omega\right)\right) && \text{[bare \ensuremath{\iota}̃\_\ensuremath{\omega}(V) inside Act(D, \_) → Act(D, MultiEval(V, \ensuremath{\omega}))  (V deg 1)]\,(axiom)}\;\text{--- apply axiom: bare \ensuremath{\iota}̃\_\ensuremath{\omega}(V) inside Act(D, \_) → Act(D, MultiEval(V, \ensuremath{\omega}))  (V deg 1)} \\
\left(\tilde{d}\!\left(V\!\left(\omega\right)\right) + \tilde{\iota}_{\omega}\!\left(\tilde{d}\!\left(V\right)\right)\right)\!\left(\eta\right) &\to \left(\tilde{d}\!\left(V\!\left(\omega\right)\right)\right)\!\left(\eta\right) + \left(\tilde{\iota}_{\omega}\!\left(\tilde{d}\!\left(V\right)\right)\right)\!\left(\eta\right) && \text{[MultiEval head linearity]\,(axiom)}\;\text{--- apply axiom: MultiEval head linearity} \\
\left(\tilde{d}\!\left(V\!\left(\omega\right)\right)\right)\!\left(\eta\right) &\to \left(\pi\sharp\!\left(\eta\right)\right)\!\left(V\!\left(\omega\right)\right) && \text{[d̃ intrinsic (Koszul) [\ensuremath{\pi}]: (d̃V)(\ensuremath{\eta}\_0, …) = \ensuremath{\Sigma} ±\ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\eta}\_i)\ensuremath{\cdot}V(…) + \ensuremath{\Sigma} ±V([\ensuremath{\eta}\_i,\ensuremath{\eta}\_j]\_K, …)]\,(axiom)}\;\text{--- apply axiom: d̃ intrinsic (Koszul) [\ensuremath{\pi}]: (d̃V)(\ensuremath{\eta}\_0, …) = \ensuremath{\Sigma} ±\ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\eta}\_i)\ensuremath{\cdot}V(…) + \ensuremath{\Sigma} ±V([\ensuremath{\eta}\_i,\ensuremath{\eta}\_j]\_K, …)} \\
\left(\tilde{\iota}_{\omega}\!\left(\tilde{d}\!\left(V\right)\right)\right)\!\left(\eta\right) &\to \left(\tilde{d}\!\left(V\right

## 5. İlişki 4: $[\tilde{\mathcal{L}}_\alpha, \tilde{\iota}_\beta] = \tilde{\iota}_{[\alpha,\beta]_K}$

$$
\tilde{\mathcal{L}}_\alpha\bigl(\tilde{\iota}_\beta V\bigr)
   \;-\; \tilde{\iota}_\beta\bigl(\tilde{\mathcal{L}}_\alpha V\bigr)
   \;=\; \tilde{\iota}_{[\alpha,\beta]_K}\,V.
$$

**Yapı.** $V$ 2-vector seçilir; $\tilde{\iota}_\beta V$ 1-vector,
tek 1-form $\eta$ ile değerlendirildiğinde fonksiyona düşer. Sağdaki
Koszul bracket $[\alpha,\beta]_K$, `KoszulProblem`'in kayıt ettiği
bracket-açılım kuralıyla Lichnerowicz Cartan formuna açılır:
$\mathcal{L}_{\rho\alpha}\beta - \mathcal{L}_{\rho\beta}\alpha -
d\langle\rho\alpha,\beta\rangle$. Sonra $\tilde{\iota}$-doğrusallığı
katsayı katsayı dağılır ve engine kapanışı tamamlar.


In [6]:
prob, _, _, V, alpha, beta = _build(form_names=("α", "β"), V_degree=2)
eta = Symbol("η")
prob.registry.declare(eta, Graded(degree=1))

lhs = Sum.make(
    Act(TildeLieDerivative(alpha, prob.pi),
        Act(TildeInteriorProduct(beta), V)),
    Neg(Act(TildeInteriorProduct(beta),
            Act(TildeLieDerivative(alpha, prob.pi), V))),
)
bracket_form = prob.bracket(alpha, beta)
rhs = Act(TildeInteriorProduct(bracket_form), V)
chain = prove("(2-VF, eval η) [L̃_α, ι̃_β] V = ι̃_[α,β]_K V",
              prob, lhs, rhs, etas=(eta,))
display_chain(chain)


(2-VF, eval η) [L̃_α, ι̃_β] V = ι̃_[α,β]_K V → 22 adımda kapandı


\begin{align*}
\left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{\iota}_{\beta}\!\left(V\right)\right) - \tilde{\iota}_{\beta}\!\left(\tilde{\mathcal{L}}_{\alpha}\!\left(V\right)\right)\right)\!\left(\eta\right) &\to \left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\right)\!\left(\eta\right) - \left(\tilde{\iota}_{\beta}\!\left(\tilde{\mathcal{L}}_{\alpha}\!\left(V\right)\right)\right)\!\left(\eta\right) && \text{[MultiEval head linearity]\,(axiom)}\;\text{--- apply axiom: MultiEval head linearity} \\
\left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\right)\!\left(\eta\right) &\to \left(\pi\sharp\!\left(\alpha\right)\right)\!\left(\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\!\left(\eta\right)\right) - \left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\!\left(\left[\alpha,\, \eta\right]_{[\cdot,\cdot]_K[\pi]}\right) && \text{[L̃\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (L̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, …) − \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, …, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, …)]\,(axiom)}\;\text{--- apply axiom: L̃\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (L̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, …) − \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, …, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, …)} \\
\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\!\left(\eta\right) &\to V\!\left(\beta,\, \eta\right) && \text{[\ensuremath{\iota}̃\_\ensuremath{\omega} intrinsic: (\ensuremath{\iota}̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, …)]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}̃\_\ensuremath{\omega} intrinsic: (\ensuremath{\iota}̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, …)} \\
\left[\alpha,\, \eta\right]_{[\cdot,\cdot]_K[\pi]} &\to L_\pi\sharp(\alpha)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\alpha\right) - d\!\left(\langle \pi\sharp\!\left(\alpha\right),\, \eta \rangle\right) && \text{[[\ensuremath{\alpha}, \ensuremath{\beta}]\_K = L\_\ensuremath{\rho}\ensuremath{\alpha}(\ensuremath{\beta}) − L\_\ensuremath{\rho}\ensuremath{\beta}(\ensuremath{\alpha}) − d\ensuremath{\langle}\ensuremath{\rho}\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} [[\ensuremath{\cdot},\ensuremath{\cdot}]\_K[\ensuremath{\pi}]]]\,(axiom)}\;\text{--- apply axiom: [\ensuremath{\alpha}, \ensuremath{\beta}]\_K = L\_\ensuremath{\rho}\ensuremath{\alpha}(\ensuremath{\beta}) − L\_\ensuremath{\rho}\ensuremath{\beta}(\ensuremath{\alpha}) − d\ensuremath{\langle}\ensuremath{\rho}\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} [[\ensuremath{\cdot},\ensuremath{\cdot}]\_K[\ensuremath{\pi}]]} \\
\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\!\left(L_\pi\sharp(\alpha)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\alpha\right) - d\!\left(\langle \pi\sharp\!\left(\alpha\right),\, \eta \rangle\right)\right) &\to V\!\left(\beta,\, L_\pi\sharp(\alpha)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\alpha\right) - d\!\left(\langle \pi\sharp\!\left(\alpha\right),\, \eta \rangle\right)\right) && \text{[\ensuremath{\iota}̃\_\ensuremath{\omega} intrinsic: (\ensuremath{\iota}̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, …)]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}̃\_\ensuremath{\omega} intrinsic: (\ensuremath{\iota}̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, …)} \\
V\!\left(\beta,\, L_\pi\sharp(\alpha)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\alpha\right) - d\!\left(\langle \pi\sharp\!\left(\alpha\right),\, \eta \rangle\right)\right) &\to V\!\left(\beta,\, L_\pi\s

## 6. İlişki 5: $[\tilde{\mathcal{L}}_\alpha, \tilde{d}] = 0$ (Poisson)

$$
\tilde{\mathcal{L}}_\alpha\bigl(\tilde{d} V\bigr)
   \;-\; \tilde{d}\bigl(\tilde{\mathcal{L}}_\alpha V\bigr)
   \;=\; 0
\qquad \bigl([\pi,\pi]_{\mathrm{SN}} = 0\bigr).
$$

**Yapı (14.G kapanışı).** 1-vector $V$ + iki 1-form $(\eta,\xi)$ ile
değerlendirildiğinde fark başlangıçta yaklaşık 24 SN-Jacobi terimine
çıkar. Engine pipeline'ı:

1. **Slot-Lie commutator** + **anchor Lie homomorphism** + **pairing
   Leibniz** kuralları residue'yu 9 terime düşürür.
2. `WrappedPairingAnchorAntisymmetryDefinition` ortak `Act`/`MultiEval`
   sarmalı altında $\langle \pi^\sharp a, b\rangle +
   \langle \pi^\sharp b, a\rangle = 0$ ile gelen 2 ikili iptali yakalar
   → 5 terim.
3. `TildeSnJacobiResidueDefinition` kalan 5-terimli formu Schouten–
   Nijenhuis Jacobi engelinin kanonik formu olarak tanır ve Poisson
   altında sıfırlar.

`prob.assume_poisson()` her iki kuralı aktive eder.


In [7]:
prob, _, _, V, alpha, eta = _build(form_names=("α", "η"), V_degree=1)
prob.assume_poisson()
xi = Symbol("ξ")
prob.registry.declare(xi, Graded(degree=1))

lhs = Sum.make(
    Act(TildeLieDerivative(alpha, prob.pi),
        Act(TildeExteriorDerivative(prob.pi), V)),
    Neg(Act(TildeExteriorDerivative(prob.pi),
            Act(TildeLieDerivative(alpha, prob.pi), V))),
)
chain = prove("(1-VF, eval η,ξ) [L̃_α, d̃] V = 0  [π Poisson]",
              prob, lhs, Integer(0), etas=(eta, xi))
display_chain(chain)


(1-VF, eval η,ξ) [L̃_α, d̃] V = 0  [π Poisson] → 133 adımda kapandı


\begin{align*}
\left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{d}\!\left(V\right)\right) - \tilde{d}\!\left(\tilde{\mathcal{L}}_{\alpha}\!\left(V\right)\right)\right)\!\left(\eta,\, \xi\right) &\to \left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{d}\!\left(V\right)\right)\right)\!\left(\eta,\, \xi\right) - \left(\tilde{d}\!\left(\tilde{\mathcal{L}}_{\alpha}\!\left(V\right)\right)\right)\!\left(\eta,\, \xi\right) && \text{[MultiEval head linearity]\,(axiom)}\;\text{--- apply axiom: MultiEval head linearity} \\
\left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{d}\!\left(V\right)\right)\right)\!\left(\eta,\, \xi\right) &\to \left(\pi\sharp\!\left(\alpha\right)\right)\!\left(\left(\tilde{d}\!\left(V\right)\right)\!\left(\eta,\, \xi\right)\right) - \left(\tilde{d}\!\left(V\right)\right)\!\left(\left[\alpha,\, \eta\right]_{[\cdot,\cdot]_K[\pi]},\, \xi\right) - \left(\tilde{d}\!\left(V\right)\right)\!\left(\eta,\, \left[\alpha,\, \xi\right]_{[\cdot,\cdot]_K[\pi]}\right) && \text{[L̃\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (L̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, …) − \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, …, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, …)]\,(axiom)}\;\text{--- apply axiom: L̃\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (L̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, …) − \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, …, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, …)} \\
\left(\tilde{d}\!\left(V\right)\right)\!\left(\eta,\, \xi\right) &\to \left(\pi\sharp\!\left(\eta\right)\right)\!\left(V\!\left(\xi\right)\right) - \left(\pi\sharp\!\left(\xi\right)\right)\!\left(V\!\left(\eta\right)\right) - V\!\left(\left[\eta,\, \xi\right]_{[\cdot,\cdot]_K[\pi]}\right) && \text{[d̃ intrinsic (Koszul) [\ensuremath{\pi}]: (d̃V)(\ensuremath{\eta}\_0, …) = \ensuremath{\Sigma} ±\ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\eta}\_i)\ensuremath{\cdot}V(…) + \ensuremath{\Sigma} ±V([\ensuremath{\eta}\_i,\ensuremath{\eta}\_j]\_K, …)]\,(axiom)}\;\text{--- apply axiom: d̃ intrinsic (Koszul) [\ensuremath{\pi}]: (d̃V)(\ensuremath{\eta}\_0, …) = \ensuremath{\Sigma} ±\ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\eta}\_i)\ensuremath{\cdot}V(…) + \ensuremath{\Sigma} ±V([\ensuremath{\eta}\_i,\ensuremath{\eta}\_j]\_K, …)} \\
\left[\eta,\, \xi\right]_{[\cdot,\cdot]_K[\pi]} &\to L_\pi\sharp(\eta)\!\left(\xi\right) - L_\pi\sharp(\xi)\!\left(\eta\right) - d\!\left(\langle \pi\sharp\!\left(\eta\right),\, \xi \rangle\right) && \text{[[\ensuremath{\alpha}, \ensuremath{\beta}]\_K = L\_\ensuremath{\rho}\ensuremath{\alpha}(\ensuremath{\beta}) − L\_\ensuremath{\rho}\ensuremath{\beta}(\ensuremath{\alpha}) − d\ensuremath{\langle}\ensuremath{\rho}\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} [[\ensuremath{\cdot},\ensuremath{\cdot}]\_K[\ensuremath{\pi}]]]\,(axiom)}\;\text{--- apply axiom: [\ensuremath{\alpha}, \ensuremath{\beta}]\_K = L\_\ensuremath{\rho}\ensuremath{\alpha}(\ensuremath{\beta}) − L\_\ensuremath{\rho}\ensuremath{\beta}(\ensuremath{\alpha}) − d\ensuremath{\langle}\ensuremath{\rho}\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} [[\ensuremath{\cdot},\ensuremath{\cdot}]\_K[\ensuremath{\pi}]]} \\
V\!\left(L_\pi\sharp(\eta)\!\left(\xi\right) - L_\pi\sharp(\xi)\!\left(\eta\right) - d\!\left(\langle \pi\sharp\!\left(\eta\right),\, \xi \rangle\right)\right) &\to V\!\left(L_\pi\sharp(\eta)\!\left(\xi\right)\right) - V\!\left(L_\pi\sharp(\xi)\!\left(\eta\right)\right) - V\!\left(d\!\left(\langle \pi\sharp\!\left(\eta\right),\, \xi \rangle\right)\right) && \text{[MultiEval arg slot linearity]\,(axiom)}\;\text{--- apply axiom: MultiEval arg slot linearity} \\
\left[\alpha,\, \eta\right]_{[\cdot,\cdot]_K[\pi]} &\to L_\pi\sharp(\alpha)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\alpha\rig

## 7. İlişki 6: $[\tilde{\mathcal{L}}_\alpha, \tilde{\mathcal{L}}_\beta] = \tilde{\mathcal{L}}_{[\alpha,\beta]_K}$ (Poisson)

$$
\tilde{\mathcal{L}}_\alpha\bigl(\tilde{\mathcal{L}}_\beta V\bigr)
   \;-\; \tilde{\mathcal{L}}_\beta\bigl(\tilde{\mathcal{L}}_\alpha V\bigr)
   \;=\; \tilde{\mathcal{L}}_{[\alpha,\beta]_K}\,V
\qquad \bigl([\pi,\pi]_{\mathrm{SN}} = 0\bigr).
$$

**Yapı (14.G kapanışı).** İlişki 5 ile aynı pipeline:
slot-Lie/anchor-Lie/pairing-Leibniz açılımları 9 terimlik residue
bırakır, `WrappedPairingAnchorAntisymmetryDefinition` 2 ikili iptal
ile 5 terime indirir, `TildeSnJacobiResidueDefinition` SN-Jacobi
engelini kapatır. 1-vector $V$ ve tek 1-form $\eta$ üzerinde
fonksiyona düşer.


In [8]:
prob, _, _, V, alpha, beta = _build(form_names=("α", "β"), V_degree=1)
prob.assume_poisson()
eta = Symbol("η")
prob.registry.declare(eta, Graded(degree=1))

lhs = Sum.make(
    Act(TildeLieDerivative(alpha, prob.pi),
        Act(TildeLieDerivative(beta, prob.pi), V)),
    Neg(Act(TildeLieDerivative(beta, prob.pi),
            Act(TildeLieDerivative(alpha, prob.pi), V))),
)
bracket_form = prob.bracket(alpha, beta)
rhs = Act(TildeLieDerivative(bracket_form, prob.pi), V)
chain = prove("(1-VF, eval η) [L̃_α, L̃_β] V = L̃_[α,β]_K V  [π Poisson]",
              prob, lhs, rhs, etas=(eta,))
display_chain(chain)


(1-VF, eval η) [L̃_α, L̃_β] V = L̃_[α,β]_K V  [π Poisson] → 119 adımda kapandı


\begin{align*}
\left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{\mathcal{L}}_{\beta}\!\left(V\right)\right) - \tilde{\mathcal{L}}_{\beta}\!\left(\tilde{\mathcal{L}}_{\alpha}\!\left(V\right)\right)\right)\!\left(\eta\right) &\to \left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{\mathcal{L}}_{\beta}\!\left(V\right)\right)\right)\!\left(\eta\right) - \left(\tilde{\mathcal{L}}_{\beta}\!\left(\tilde{\mathcal{L}}_{\alpha}\!\left(V\right)\right)\right)\!\left(\eta\right) && \text{[MultiEval head linearity]\,(axiom)}\;\text{--- apply axiom: MultiEval head linearity} \\
\left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{\mathcal{L}}_{\beta}\!\left(V\right)\right)\right)\!\left(\eta\right) &\to \left(\pi\sharp\!\left(\alpha\right)\right)\!\left(\left(\tilde{\mathcal{L}}_{\beta}\!\left(V\right)\right)\!\left(\eta\right)\right) - \left(\tilde{\mathcal{L}}_{\beta}\!\left(V\right)\right)\!\left(\left[\alpha,\, \eta\right]_{[\cdot,\cdot]_K[\pi]}\right) && \text{[L̃\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (L̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, …) − \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, …, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, …)]\,(axiom)}\;\text{--- apply axiom: L̃\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (L̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, …) − \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, …, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, …)} \\
\left(\tilde{\mathcal{L}}_{\beta}\!\left(V\right)\right)\!\left(\eta\right) &\to \left(\pi\sharp\!\left(\beta\right)\right)\!\left(V\!\left(\eta\right)\right) - V\!\left(\left[\beta,\, \eta\right]_{[\cdot,\cdot]_K[\pi]}\right) && \text{[L̃\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (L̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, …) − \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, …, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, …)]\,(axiom)}\;\text{--- apply axiom: L̃\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (L̃\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, …) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, …) − \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, …, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, …)} \\
\left[\beta,\, \eta\right]_{[\cdot,\cdot]_K[\pi]} &\to L_\pi\sharp(\beta)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\beta\right) - d\!\left(\langle \pi\sharp\!\left(\beta\right),\, \eta \rangle\right) && \text{[[\ensuremath{\alpha}, \ensuremath{\beta}]\_K = L\_\ensuremath{\rho}\ensuremath{\alpha}(\ensuremath{\beta}) − L\_\ensuremath{\rho}\ensuremath{\beta}(\ensuremath{\alpha}) − d\ensuremath{\langle}\ensuremath{\rho}\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} [[\ensuremath{\cdot},\ensuremath{\cdot}]\_K[\ensuremath{\pi}]]]\,(axiom)}\;\text{--- apply axiom: [\ensuremath{\alpha}, \ensuremath{\beta}]\_K = L\_\ensuremath{\rho}\ensuremath{\alpha}(\ensuremath{\beta}) − L\_\ensuremath{\rho}\ensuremath{\beta}(\ensuremath{\alpha}) − d\ensuremath{\langle}\ensuremath{\rho}\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} [[\ensuremath{\cdot},\ensuremath{\cdot}]\_K[\ensuremath{\pi}]]} \\
V\!\left(L_\pi\sharp(\beta)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\beta\right) - d\!\left(\langle \pi\sharp\!\left(\beta\right),\, \eta \rangle\right)\right) &\to V\!\left(L_\pi\sharp(\beta)\!\left(\eta\right)\right) - V\!\left(L_\pi\sharp(\eta)\!\left(\beta\right)\right) - V\!\left(d\!\left(\langle \pi\sharp\!\left(\beta\right),\, \eta \rangle\right)\right) && \text{[MultiEval arg slot linearity]\,(axiom)}\;\text{--- apply axiom: MultiEval arg slot linearity} \\
\left[\alpha,\, \eta\right]_{[\cdot,\cdot]_K[\pi]} &\to L_\pi\sharp(\alpha)\!\left(\eta\

## Sonuç

Altı tilde Cartan ilişkisi de jenerik bir multivector $V$ üzerinde,
tek bir `prob.prove_tilde_cartan` çağrısıyla syntactic olarak kapandı.

| # | İlişki | $V$ derecesi | Eval | Poisson? | Kapanış kanalı |
|---|---|---|---|---|---|
| 1 | $\tilde{\iota}\tilde{\iota}$ anti-commute | 2 | $\xi$ | hayır | MultiEval alternating |
| 2 | $\tilde{d}^2 V = 0$ | 1 | $\eta,\xi$ | **evet** | `TildeDSquaredPoisson` (Aux-5) |
| 3 | Cartan magic | 1 | $\eta$ | hayır | `TildeCartanMagic` (14.F) |
| 4 | $[\tilde{\mathcal{L}}, \tilde{\iota}]$ commutator | 2 | $\eta$ | hayır | bracket-açılım + intrinsik |
| 5 | $[\tilde{\mathcal{L}}, \tilde{d}] = 0$ | 1 | $\eta,\xi$ | **evet** | 14.G: WrappedPairingAntisym + SN-Jacobi recognizer |
| 6 | $[\tilde{\mathcal{L}}, \tilde{\mathcal{L}}]$ commutator | 1 | $\eta$ | **evet** | 14.G: aynı pipeline |

İspatların tamamı 28-kurallı tek engine üzerinden çalışıyor; her adımın
hangi `Definition`'dan geldiği `ProofStep.rule` alanında saklanır ve
`display_chain` LaTeX olarak okunabilir biçimde basar.
